# Central and Eastern Europe agriculture climate analysis

By [Ben Welsh](http://palewi.re/who-is-ben-welsh/)

This repository contains data and code supporting a Reuters analysis of June and
July temperatures in Hungary, Romania and Poland.

Published September 2, 2026: [Climate shocks hit EU maize heartland in Hungary,
Romania](https://www.reuters.com/business/environment/climate-shocks-hit-eu-maize-heartland-hungary-romania-2026-09-02/).

The key finding:

> "Over the past decade, Hungary's June and July highs have exceeded the 1961 to 1990 average by ​3.1 degrees Celsius (5.6 degrees Fahrenheit), according to [Reuters Climate Monitor](https://www.reuters.com/graphics/CLIMATE-AUTOMATED/MONITOR/akpeykqqapr/) data. Romania is close behind, with average summer highs 3 C (5.4 F) above historical norms."



# Configuration

Import Python tools and environment variables.

In [6]:
import os
from io import BytesIO

import boto3
import dotenv
import pandas as pd
import altair as alt

In [7]:
env_path = dotenv.find_dotenv(usecwd=True)
dotenv.load_dotenv(env_path)

True

Set the key variables for analysis.

In [8]:
COUNTY_LIST: set[str] = {
    "Romania",
    "Hungary",
    "Poland"
}
VARIABLE: str = "t2m_max"

# Data extraction

The underlying data are [ERA5](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview) reanalysis estimates produced by the [Reuters
Climate Monitor](https://www.reuters.com/graphics/CLIMATE-AUTOMATED/MONITOR/akpeykqqapr/).

The notebook downloads daily area-weighted country averages from the private Reuters dataset at `analysis/daily-country-averages/era5.parquet`. It averages `t2m_max` from June
1 through July 31 for each country and year, then subtracts that country's
1961–1990 average for the same 61 calendar days.

In [9]:
def get_era5_country_averages() -> pd.DataFrame:
    # Connect to S3
    s3 = boto3.client("s3")

    # Link to the Reuters Climate Monitor bucket
    bucket: str = os.getenv("S3_BUCKET_NAME")
    assert bucket

    # Download daily area-weighted country averages for the variable of interest
    response = s3.get_object(Bucket=bucket, Key="analysis/daily-country-averages/era5.parquet")
    daily = pd.read_parquet(
        BytesIO(response["Body"].read()),
        columns=["country", "date", VARIABLE],
    )

    # Set the date column to datetime and filter for the countries of interest
    daily["date"] = pd.to_datetime(daily["date"])
    daily["year"] = daily["date"].dt.year
    daily = daily[daily["country"].isin(COUNTY_LIST)]

    # Return the daily DataFrame
    return daily

In [10]:
daily_df: pd.DataFrame = get_era5_country_averages()

In [11]:
daily_df.head()

,country,date,t2m_max,year
175,Poland,1950-01-01,-1.869833,1950
181,Romania,1950-01-01,-4.821524,1950
97,Hungary,1950-01-01,-1.280286,1950
175,Poland,1950-01-02,0.188459,1950
181,Romania,1950-01-02,-2.626997,1950


Filter to the 1961–1990 baseline period and all days in June and July.

In [12]:
baseline_df: pd.DataFrame = daily_df.loc[
    (daily_df["year"].between(1961, 1990)) &
    (daily_df["date"].dt.month.isin([6, 7]))
].copy()

In [13]:
baseline_df.head()

,country,date,t2m_max,year
181,Romania,1961-06-01,23.276005,1961
175,Poland,1961-06-01,23.378082,1961
97,Hungary,1961-06-01,25.792429,1961
175,Poland,1961-06-02,24.877268,1961
181,Romania,1961-06-02,25.000109,1961


Verify that we have complete data for the baseline period.

In [14]:
# Make sure there are 30 unique years
assert baseline_df["year"].nunique() == 30

In [15]:
# Full date coverage for the baseline period
assert baseline_df["date"].nunique() == 61 * 30

In [16]:
# No missing values
assert baseline_df.isna().sum().sum() == 0

Calculate the average high temperature for each country in the baseline period.

In [17]:
baseline_mean: pd.DataFrame = (
    baseline_df.groupby("country")[VARIABLE]
        .mean()
        .reset_index()
)

In [18]:
baseline_mean.head()

,country,t2m_max
0,Hungary,24.976462
1,Poland,20.930368
2,Romania,23.660751


Calculate the annual average high temperature for each country in June and July. Compare it to the baseline average to calculate the anomaly for each year.

In [19]:
analysis_df: pd.DataFrame = (
    # Filter to June and July in our study period
    daily_df.loc[
        (daily_df["year"].between(1961, 2026)) &
        (daily_df["date"].dt.month.isin([6, 7]))
    ].copy()
    # Calculate the annual average high temperature for each country in each year
     .groupby(["country", "year"])[VARIABLE]
     .mean()
     .reset_index()
     # Merge the annual averages with the baseline averages
     .merge(baseline_mean, on=["country"], how="inner", validate="many_to_one", suffixes=("", "_baseline"))
     # Calculate the anomaly for each year by subtracting the baseline average from the annual average
     .assign(anomaly=lambda x: x[VARIABLE] - x[f"{VARIABLE}_baseline"])
)

In [20]:
analysis_df.head()

,country,year,t2m_max,t2m_max_baseline,anomaly
0,Hungary,1961,26.077559,24.976462,1.101097
1,Hungary,1962,23.889714,24.976462,-1.086748
2,Hungary,1963,27.630426,24.976462,2.653964
3,Hungary,1964,27.343266,24.976462,2.366804
4,Hungary,1965,23.935210,24.976462,-1.041252


## Findings

Sketch the results

In [21]:
alt.Chart(analysis_df).mark_bar().encode(
    x=alt.X(
        'year:Q',
        title='Year',
        axis=alt.Axis(
            format="d",
            tickMinStep=5,
        ),
    ),
    y='anomaly:Q',
    facet='country:N',
    color='country:N'
 ).properties(width=300)

alt.Chart(...)

Pivot to wide data for charting in Datawrapper.

In [22]:
pivot_df: pd.DataFrame = analysis_df.pivot(
    index='year',
    columns='country',
    values='anomaly'
).round(3)

In [23]:
pivot_df.tail(10)

country,Hungary,Poland,Romania
year,,,
2017,3.005,1.412,2.666
2018,1.626,2.883,1.339
2019,3.236,4.055,2.705
2020,0.861,1.669,1.587
2021,4.299,4.161,2.812
2022,4.302,2.928,3.725
2023,2.379,2.635,2.509
2024,3.828,3.299,5.428
2025,3.582,2.083,4.350


Output as a CSV.

In [24]:
pivot_df.to_csv("pivot.csv", index=True)

Calculate the average anomaly over the past decade for our key finding.

In [28]:
(
    pivot_df.tail(10)
        .mean()
        .round(1)
        .reset_index()
)

,country,0
0,Hungary,3.1
1,Poland,2.8
2,Romania,3.0
